# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install monai nibabel huggingface_hub -q

import torch
print("CUDA available:", torch.cuda.is_available())

import sys
sys.path.append('/content/drive/MyDrive/knee-segmentation-cv/src')

print("Setup complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: True
Setup complete


In [ ]:
# !pip install huggingface_hub -q

# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="YongchengYAO/OAIZIB-CM",
#     repo_type="dataset",
#     local_dir="/content/drive/MyDrive/knee-segmentation-cv/data/raw"
# )

# Train

## Preprocessing

In [ ]:
import os

processed_drive_dir = "/content/drive/MyDrive/knee-segmentation-cv/data/processed"
processed_local_dir = "/content/data/processed"

os.makedirs(processed_drive_dir, exist_ok=True)

if os.listdir(processed_drive_dir):
    print("Found preprocessed data on Drive, syncing down...")
    os.makedirs(processed_local_dir, exist_ok=True)
    !rsync -a {processed_drive_dir}/ {processed_local_dir}/
else:
    print("No preprocessed data found, syncing raw data and running preprocessing once (this can take a while)...")
    !mkdir -p /content/data/raw
    !rsync -a /content/drive/MyDrive/knee-segmentation-cv/data/raw/imagesTr/ /content/data/raw/imagesTr/
    !rsync -a /content/drive/MyDrive/knee-segmentation-cv/data/raw/labelsTr/ /content/data/raw/labelsTr/

    from preprocess import main as preprocess_main
    preprocess_main(config_path="/content/drive/MyDrive/knee-segmentation-cv/configs/config.yaml")

    print("Syncing processed data up to Drive for future sessions...")
    !rsync -a {processed_local_dir}/ {processed_drive_dir}/

print("Preprocessing setup complete")

Found preprocessed data on Drive, syncing down...
Preprocessing setup complete


In [ ]:
from train import main
main(config_path="/content/drive/MyDrive/knee-segmentation-cv/configs/config.yaml", resume=True)

Using device: cuda
Found 404 processed pairs of image/mask
Checkpoint loaded: epoch=12, metric=0.8782
Resuming from epoch 13


Train:   0%|          | 0/3430 [00:00<?, ?it/s]

Val:   0%|          | 0/610 [00:00<?, ?it/s]

Epoch 13: train_loss=0.0860 val_loss=0.0781 val_dice=0.8796
Results saved to /content/drive/MyDrive/knee-segmentation-cv/results/metrics/train_history.json


Train:   0%|          | 0/3430 [00:00<?, ?it/s]

Val:   0%|          | 0/610 [00:00<?, ?it/s]

... (6 outputs left out)

Epoch 16: train_loss=0.0798 val_loss=0.0746 val_dice=0.8831
Results saved to /content/drive/MyDrive/knee-segmentation-cv/results/metrics/train_history.json


Train:   0%|          | 0/3430 [00:00<?, ?it/s]

Val:   0%|          | 0/610 [00:00<?, ?it/s]

Epoch 17: train_loss=0.0788 val_loss=0.0731 val_dice=0.8817
Results saved to /content/drive/MyDrive/knee-segmentation-cv/results/metrics/train_history.json
Early stopping triggered at epoch 17


# Test

In [2]:
import os

local_ts_images = "/content/data/raw/imagesTs"
local_ts_labels = "/content/data/raw/labelsTs"

if not os.path.exists(local_ts_images):
    print("Syncing test set (imagesTs/labelsTs) from Drive...")
    !mkdir -p /content/data/raw
    !rsync -a /content/drive/MyDrive/knee-segmentation-cv/data/raw/imagesTs/ /content/data/raw/imagesTs/
    !rsync -a /content/drive/MyDrive/knee-segmentation-cv/data/raw/labelsTs/ /content/data/raw/labelsTs/
else:
    print("Test set already present locally")

print("Test data ready")

Test set already present locally
Test data ready


In [4]:
from evaluate import evaluate

results = evaluate(
    config_path="/content/drive/MyDrive/knee-segmentation-cv/configs/config.yaml",
    results_name="test_evaluation_best",
)

Found 103 pairs of image/mask
Model loaded from /content/drive/MyDrive/knee-segmentation-cv/checkpoints/best.pth (epoch=7, val_dice=0.8919)


Evaluate:   0%|          | 0/1030 [00:00<?, ?it/s]


=== Results on test ===
  Dice [femur]: 0.9735
  Dice [femoral_cartilage]: 0.8731
  Dice [tibia]: 0.9700
  Dice [medial_tibial_cartilage]: 0.8125
  Dice [lateral_tibial_cartilage]: 0.8384
  Mean Dice: 0.8935
 Hausdorff95 [femur]: 3.1068
 Hausdorff95 [femoral_cartilage]: 3.7898
 Hausdorff95 [tibia]: 2.3893
 Hausdorff95 [medial_tibial_cartilage]: 3.7309
 Hausdorff95 [lateral_tibial_cartilage]: 3.2277
Results saved to /content/drive/MyDrive/knee-segmentation-cv/results/metrics/test_evaluation_best.json


In [3]:
from evaluate import evaluate

results_last = evaluate(
    config_path="/content/drive/MyDrive/knee-segmentation-cv/configs/config.yaml",
    checkpoint_path="/content/drive/MyDrive/knee-segmentation-cv/checkpoints/last.pth",
    results_name="test_evaluation_last",
)

Found 103 pairs of image/mask
Model loaded from /content/drive/MyDrive/knee-segmentation-cv/checkpoints/last.pth (epoch=17, val_dice=0.8817)


Evaluate:   0%|          | 0/1030 [00:00<?, ?it/s]


=== Results on test ===
  Dice [femur]: 0.9697
  Dice [femoral_cartilage]: 0.8628
  Dice [tibia]: 0.9665
  Dice [medial_tibial_cartilage]: 0.7936
  Dice [lateral_tibial_cartilage]: 0.8199
  Mean Dice: 0.8825
 Hausdorff95 [femur]: 2.9694
 Hausdorff95 [femoral_cartilage]: 3.9365
 Hausdorff95 [tibia]: 2.2671
 Hausdorff95 [medial_tibial_cartilage]: 3.2811
 Hausdorff95 [lateral_tibial_cartilage]: 3.0638
Results saved to /content/drive/MyDrive/knee-segmentation-cv/results/metrics/test_evaluation_last.json


# Visualize

In [ ]:
!pip install scikit-image plotly -q

import torch
import numpy as np
import nibabel as nib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from skimage.measure import marching_cubes

from predict import predict_volume
from model import build_unet
from utils import load_config, get_device

config = load_config("/content/drive/MyDrive/knee-segmentation-cv/configs/config.yaml")
device = get_device()

model = build_unet(config).to(device)
checkpoint = torch.load(f"{config['checkpoints_dir']}/best.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


image_path = "/content/data/raw/imagesTs/oaizib_445_0000.nii.gz"
case_id = image_path.split("/")[-1].replace("_0000.nii.gz", "")
mask_path = f"/content/data/raw/labelsTs/{case_id}.nii.gz"

pred_volume = predict_volume(model, image_path, device, axis=config["axis"], target_size=tuple(config["target_size"]))
gt_volume = nib.load(mask_path).get_fdata().astype(np.int64)

spacing = nib.load(image_path).header.get_zooms()

CLASS_NAMES = ["background", "femur", "femoral_cartilage", "tibia", "medial_tibial_cartilage", "lateral_tibial_cartilage"]
COLORS = {
    "femur": "whitesmoke",
    "femoral_cartilage": "royalblue",
    "tibia": "lightgray",
    "medial_tibial_cartilage": "seagreen",
    "lateral_tibial_cartilage": "darkorange",
}

def add_label_meshes(fig, volume, row, col, show_legend):
    for label in range(1, len(CLASS_NAMES)):
        mask = volume == label
        if not mask.any():
            continue

        verts, faces, _, _ = marching_cubes(mask.astype(np.float32), level=0.5, spacing=spacing)
        x, y, z = verts.T
        i, j, k = faces.T
        name = CLASS_NAMES[label]

        fig.add_trace(
            go.Mesh3d(
                x=x, y=y, z=z, i=i, j=j, k=k,
                color=COLORS[name], opacity=1.0,
                name=name, showlegend=show_legend, flatshading=True,
                legendgroup=name,
            ),
            row=row, col=col,
        )

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=["Prediction", "Ground Truth"],
)

add_label_meshes(fig, pred_volume, row=1, col=1, show_legend=True)
add_label_meshes(fig, gt_volume, row=1, col=2, show_legend=False)

fig.update_scenes(aspectmode="data")
fig.update_layout(
    title=f"Predicted vs Ground Truth ({case_id})",
    legend=dict(itemsizing="constant"),
    width=1200, height=700,
)
fig.show()